# 04: silver labels, scoring and holdout check

Inputs: competition data, outputs of 03b_label_shard0 and 03b_label_shard1.
Output: silver_scores.csv (one score per study and finding) and silver_meta.json.

Scoring (decided on dev in 03):
- severity > 0: score = severity + 0.25 * certainty
- severity 0 with a verbatim quote: score 0 (explicit normal)
- unmentioned, or severity 0 with empty evidence (D1) or evidence not found in the report (D3): 0.5

Rule fixed before running this notebook:
- D4 (Synovitis, Contusion and Fracture: treat severity 0 as unmentioned) is adopted only if
  the mean dev AUC of those three labels does not go down.
- Holdout is scored once, after D4 is decided. Nothing in the scoring changes because of it.

# Cell 1: merge shards, rebuild the gold split

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

def find_all(pattern):
    hits = []
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

paths = find_all("labels_shard*.csv")
print(paths)
lab = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
train = pd.read_csv(f"{ROOT}/train.csv")
print("rows:", len(lab), " unique studies:", lab["StudyInstanceUID"].nunique(), " train:", len(train))
print("train studies without labels:", len(set(train["StudyInstanceUID"]) - set(lab["StudyInstanceUID"])))
print(lab["status"].value_counts().to_string())

gold = train[train[LABELS].notna().all(axis=1)].reset_index(drop=True)
rng = np.random.default_rng(42)
dev_ids = set(rng.choice(gold["StudyInstanceUID"], size=20, replace=False))
gold["split"] = np.where(gold["StudyInstanceUID"].isin(dev_ids), "dev", "holdout")
print("\n" + gold.groupby("split")[LABELS].sum().astype(int).T.to_string())

# Cell 2: decide D4 on dev

In [ ]:
D4_LABELS = ["Synovitis", "Contusion", "Fracture"]

def score_table(lab, d4=False):
    out = pd.DataFrame({"StudyInstanceUID": lab["StudyInstanceUID"]})
    for k in LABELS:
        sev = lab[f"{k}|sev"]
        cert = lab[f"{k}|cert"].fillna(1)
        ev = lab[f"{k}|ev"].fillna("").astype(str).str.strip()
        found = lab[f"{k}|ev_found"].fillna(False).astype(bool)
        unmentioned = sev.isna() | ((sev == 0) & ((ev == "") | ~found))
        if d4 and k in D4_LABELS:
            unmentioned |= sev == 0
        s = (sev + 0.25 * cert).where(sev > 0, 0.0)
        out[k] = s.where(~unmentioned, 0.5)
    return out

def aucs(scores, ids):
    g = gold[gold["StudyInstanceUID"].isin(ids)].merge(scores, on="StudyInstanceUID", suffixes=("", "_s"))
    return {k: roc_auc_score(g[k], g[k + "_s"]) for k in LABELS if g[k].nunique() == 2}

dev_list = gold.loc[gold["split"] == "dev", "StudyInstanceUID"]
base = aucs(score_table(lab), dev_list)
with_d4 = aucs(score_table(lab, d4=True), dev_list)
cmp = pd.DataFrame({"D1+D3": base, "D1+D3+D4": with_d4}).round(3)
print(cmp.to_string())
print("macro:", cmp.mean().round(3).to_dict())

delta = np.mean([with_d4[k] - base[k] for k in D4_LABELS])
USE_D4 = bool(delta >= 0)
print(f"mean dev change on {D4_LABELS}: {delta:+.3f} -> USE_D4 = {USE_D4}")

# Cell 3: holdout, scored once

In [ ]:
final = score_table(lab, d4=USE_D4)
g = gold[gold["split"] == "holdout"].merge(final, on="StudyInstanceUID", suffixes=("", "_s"))
rng = np.random.default_rng(0)
B = 2000
idx = [rng.integers(0, len(g), len(g)) for _ in range(B)]

def boot_auc(y, s):
    vals = [roc_auc_score(y[i], s[i]) for i in idx if len(set(y[i])) == 2]
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

rows = {}
for k in LABELS:
    y, s = g[k].values.astype(int), g[k + "_s"].values
    if len(set(y)) == 2:
        lo, hi = boot_auc(y, s)
        rows[k] = {"pos": int(y.sum()), "auc": roc_auc_score(y, s), "lo95": lo, "hi95": hi}
res = pd.DataFrame(rows).T
print(f"holdout n = {len(g)}")
print(res.round(3).to_string())

macro_b = []
for i in idx:
    v = [roc_auc_score(g[k].values[i], g[k + "_s"].values[i]) for k in rows if len(set(g[k].values[i])) == 2]
    macro_b.append(np.mean(v))
macro = res["auc"].mean()
print(f"holdout macro AUC: {macro:.3f}  (95% CI {np.percentile(macro_b, 2.5):.3f} to {np.percentile(macro_b, 97.5):.3f})")

# Cell 4: save silver labels

In [ ]:
final.to_csv("/kaggle/working/silver_scores.csv", index=False)
meta = {
    "source_notebooks": ["03b_label_shard0", "03b_label_shard1"],
    "model": "Qwen3-8B, 4-bit NF4, greedy decoding, thinking off",
    "scoring": "sev>0: sev + 0.25*cert; explicit normal: 0; unmentioned: 0.5",
    "decisions": ["D1 empty evidence -> unmentioned", "D2 prompt v3",
                  "D3 evidence not verbatim -> unmentioned",
                  f"D4 {D4_LABELS} severity 0 -> unmentioned: {'adopted' if USE_D4 else 'rejected'}"],
    "dev_macro": round(float(np.mean(list((with_d4 if USE_D4 else base).values()))), 4),
    "holdout_macro": round(float(macro), 4),
    "holdout_macro_ci95": [round(float(np.percentile(macro_b, 2.5)), 4),
                           round(float(np.percentile(macro_b, 97.5)), 4)],
    "holdout_per_label": {k: round(float(v), 4) for k, v in res["auc"].items()},
}
json.dump(meta, open("/kaggle/working/silver_meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2))
print("\nscore distribution per finding:")
print(final[LABELS].apply(lambda s: s.round(2).value_counts()).fillna(0).astype(int).T.to_string())

# Cell 5: log metrics to W&B (no data uploaded)

In [ ]:
import hashlib

try:
    from kaggle_secrets import UserSecretsClient
    WANDB_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as err:
    WANDB_KEY = None
    print("no W&B key, skipping:", repr(err)[:120])

if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    sha = hashlib.sha256(open("/kaggle/working/silver_scores.csv", "rb").read()).hexdigest()
    run = wandb.init(project="knee-mri-abnormality", job_type="labeling", name="silver-v1",
                     config={"model": meta["model"], "prompt": "v3", "scoring": meta["scoring"],
                             "decisions": meta["decisions"], "use_d4": USE_D4,
                             "silver_scores_sha256": sha, "dev_n": 20, "holdout_n": int(len(g))})
    run.summary.update({"dev_macro": meta["dev_macro"], "holdout_macro": meta["holdout_macro"],
                        "holdout_macro_lo95": meta["holdout_macro_ci95"][0],
                        "holdout_macro_hi95": meta["holdout_macro_ci95"][1]})
    per_label = wandb.Table(columns=["finding", "holdout_pos", "holdout_auc", "lo95", "hi95"])
    for k, r in res.iterrows():
        per_label.add_data(k, int(r["pos"]), float(r["auc"]), float(r["lo95"]), float(r["hi95"]))
    history = wandb.Table(columns=["model", "setup", "prompt", "rules", "dev_macro", "est_hours"],
                          data=[["Qwen3-8B", "fp16 split over 2 GPUs", "v2", "none", 0.814, 10.2],
                                ["Qwen3-8B", "fp16 split over 2 GPUs", "v2", "D1", 0.820, 10.2],
                                ["Qwen3-4B", "fp16, one copy per GPU", "v3", "D1", 0.752, 3.0],
                                ["Qwen3-8B", "NF4, one copy per GPU", "v3", "D1", 0.811, 8.9],
                                ["Qwen3-8B", "NF4, one copy per GPU", "v3", "D1+D3", 0.814, 8.9]])
    run.log({"holdout_per_label": per_label, "labeler_runs_dev": history})
    run.finish()

## D5 (rule set before running)
For studies whose report does not mention synovitis, score = 0.5 + p, where p is the
probability that a report would mention synovitis, predicted from the other 11 findings.
Fitted on non-gold silver labels only. Adopted if the silver cross-validation AUC is at
least 0.70 and the dev Synovitis AUC goes up. Holdout is not used.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

S = final.set_index("StudyInstanceUID")
feats = [k for k in LABELS if k != "Synovitis"]
X = S[feats].copy()
for k in feats:
    X[k + " unm"] = (S[k] == 0.5).astype(float)
y = (S["Synovitis"] >= 2.0).astype(int)
is_gold = S.index.isin(set(gold["StudyInstanceUID"]))

clf = LogisticRegression(max_iter=2000)
p_cv = cross_val_predict(clf, X[~is_gold], y[~is_gold], cv=5, method="predict_proba")[:, 1]
cv_auc = roc_auc_score(y[~is_gold], p_cv)
print(f"silver check: predicting a synovitis mention from other findings, "
      f"5-fold AUC {cv_auc:.3f} ({int(y[~is_gold].sum())} positives of {int((~is_gold).sum())})")
clf.fit(X[~is_gold], y[~is_gold])
print(pd.Series(clf.coef_[0], index=X.columns).sort_values().round(2).to_string())

p = pd.Series(clf.predict_proba(X)[:, 1], index=S.index)
p[~is_gold] = p_cv
unm = S["Synovitis"] == 0.5
S2 = S.copy()
S2.loc[unm, "Synovitis"] = 0.5 + p[unm]
final_v2 = S2.reset_index()

before = aucs(final, dev_list)["Synovitis"]
after = aucs(final_v2, dev_list)["Synovitis"]
USE_D5 = bool(cv_auc >= 0.70 and after > before)
print(f"dev Synovitis AUC: {before:.3f} -> {after:.3f}  -> USE_D5 = {USE_D5}")

if USE_D5:
    final_v2.to_csv("/kaggle/working/silver_scores_v2.csv", index=False)
    print("saved silver_scores_v2.csv; silver_scores.csv is kept unchanged")